In [ ]:
import openai
openai.api_key  = ""

In [ ]:
import os

# Set the OpenAI API key as an environment variable
os.environ["OPENAI_API_KEY"] = ""

# To verify that the environment variable is set
print("OpenAI API Key:", os.environ.get("OPENAI_API_KEY"))

In [ ]:
import openai
import requests
from PIL import Image
import io
from openai import OpenAI
client = OpenAI()

def generate_image(prompt, n=1, size="1792x1024"):
   """
   Generate an image using DALL-E 3
  
   :param prompt: Text description of the image
   :param n: Number of images to generate
   :param size: Size of the image
   :return: List of image URLs
   """
   try:
       response = client.images.generate(
           model="dall-e-3",
           prompt=prompt,
           n=n,
           size=size,
           quality="standard"
       )
       urls = [img.url for img in response.data]
       print(f"Generated URLs: {urls}")  # Debug print
       return urls
   except Exception as e:
       print(f"An error occurred in generate_image: {e}")
       return []

def save_image(url, filename):
   """
   Save an image from a URL to a file
  
   :param url: URL of the image
   :param filename: Name of the file to save the image
   """
   try:
       print(f"Attempting to save image from URL: {url}")  # Debug print
       response = requests.get(url)
       response.raise_for_status()  # Raise an exception for bad status codes
       img = Image.open(io.BytesIO(response.content))
       img.save(filename)
       print(f"Image saved successfully as {filename}")
   except requests.exceptions.RequestException as e:
       print(f"Error fetching the image: {e}")
   except Exception as e:
       print(f"Error saving the image: {e}")

# Example usage
# prompt = "A futuristic city with flying cars and holographic billboards, in the style of cyberpunk anime"
prompt = "indian techie building a startup"
image_urls = generate_image(prompt)

if image_urls:
   for i, url in enumerate(image_urls):
       if url:  # Check if URL is not empty
           save_image(url, f"dalle3_image_{i+1}.png")
       else:
           print(f"Empty URL for image {i+1}")
else:
   print("No images were generated.")

In [ ]:
%cd stability_ai

## STABILITY AI

In [ ]:
# !pip install stability-sdk

In [ ]:
import os
import io
import warnings
from PIL import Image
from stability_sdk import client
import stability_sdk.interfaces.gooseai.generation.generation_pb2 as generation

# Our Host URL should not be prepended with "https" nor should it have a trailing slash.
os.environ['STABILITY_HOST'] = 'grpc.stability.ai:443'

# Sign up for an account at the following link to get an API Key.
# https://platform.stability.ai/

# Click on the following link once you have created an account to be taken to your API Key.
# https://platform.stability.ai/account/keys

# Paste your API Key below.

os.environ['STABILITY_KEY'] = ''

In [ ]:
# Set up our connection to the API.
stability_api = client.StabilityInference(
    key=os.environ['STABILITY_KEY'], # API Key reference.
    verbose=True, # Print debug messages.
    engine="stable-diffusion-xl-1024-v1-0", # Set the engine to use for generation.
    # Check out the following link for a list of available engines: https://platform.stability.ai/docs/features/api-parameters#engine
)

In [ ]:
# Set up our initial generation parameters.
answers = stability_api.generate(
    prompt="indian guy working with an amazing setup",
    seed=4253978046, # If a seed is provided, the resulting generated image will be deterministic.
                     # What this means is that as long as all generation parameters remain the same, you can always recall the same image simply by generating it again.
                     # Note: This isn't quite the case for Clip Guided generations, which we'll tackle in a future example notebook.
    steps=50, # Amount of inference steps performed on image generation. Defaults to 30. 
    cfg_scale=8.0, # Influences how strongly your generation is guided to match your prompt.
                   # Setting this value higher increases the strength in which it tries to match your prompt.
                   # Defaults to 7.0 if not specified.
    width=1024, # Generation width, defaults to 512 if not included.
    height=1024, # Generation height, defaults to 512 if not included.
    samples=1, # Number of images to generate, defaults to 1 if not included.
    sampler=generation.SAMPLER_K_DPMPP_2M # Choose which sampler we want to denoise our generation with.
                                                 # Defaults to k_dpmpp_2m if not specified. Clip Guidance only supports ancestral samplers.
                                                 # (Available Samplers: ddim, plms, k_euler, k_euler_ancestral, k_heun, k_dpm_2, k_dpm_2_ancestral, k_dpmpp_2s_ancestral, k_lms, k_dpmpp_2m, k_dpmpp_sde)
)


In [ ]:
answers

In [ ]:

# Set up our warning to print to the console if the adult content classifier is tripped.
# If adult content classifier is not tripped, save generated images.
for resp in answers:
    # print(resp)
    for artifact in resp.artifacts:
        print(artifact)
        if artifact.finish_reason == generation.FILTER:
            warnings.warn(
                "Your request activated the API's safety filters and could not be processed."
                "Please modify the prompt and try again.")
        if artifact.type == generation.ARTIFACT_IMAGE:
            img = Image.open(io.BytesIO(artifact.binary))
            img.save("test.png") # Save our generated images with their seed number as the filename.

## FLUX AI 1

In [ ]:
# !pip install -U diffusers
# !pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# !pip install transformers


In [4]:
# !huggingface-cli login

!huggingface-cli login --token ""

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /home/sarveshharikant/.cache/huggingface/token
Login successful


In [ ]:
import torch
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-dev", torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload() #save some VRAM by offloading the model to CPU. Remove this if you have enough GPU power

prompt = "A cat holding a sign that says hello world"
image = pipe(
    prompt,
    height=1024,
    width=1024,
    guidance_scale=3.5,
    num_inference_steps=50,
    max_sequence_length=512,
    generator=torch.Generator("cpu").manual_seed(0)
).images[0]
image.save("flux-dev.png")
